# OmniFile AI Processor — Colab Lab v4.2.0
**المطور:** Dr. Abdulmalek Tamer Al-husseini

**GitHub:** https://github.com/DrAbdulmalek/OmniFile_Processor

> نفّذ الخلايا بالترتيب من الأعلى.


## 0 — استنساخ المشروع وإعداد البيئة

In [ ]:
import os, sys
REPO = '/content/OmniFile_Processor'
if not os.path.exists(REPO):
    !git clone -q https://github.com/DrAbdulmalek/OmniFile_Processor.git {REPO}
else:
    !cd {REPO} && git pull -q
os.chdir(REPO)
sys.path.insert(0, REPO)
print(f'Branch: {os.popen("git branch --show-current").read().strip()}')
print(f'Commits: {os.popen("git rev-list --count HEAD").read().strip()}')


## 1 — التثبيت المرحلي

In [ ]:
print('Core packages...')
!pip install -q gradio>=4.0 Pillow numpy pandas langdetect\
              arabic-reshaper python-bidi python-docx 2>/dev/null
print('OCR engines...')
!pip install -q easyocr pytesseract PyMuPDF 2>/dev/null
!apt-get install -qq tesseract-ocr tesseract-ocr-ara 2>/dev/null
print('Done!')


In [ ]:
# Transformers (optional - needed for TrOCR and translation)
INSTALL_TRANSFORMERS = False
if INSTALL_TRANSFORMERS:
    !pip install -q transformers torch sentencepiece sacremoses jiwer
    print('Transformers ready')
else:
    print('Skipped Transformers (set INSTALL_TRANSFORMERS=True to enable)')


## 2 — فحص صحة الوحدات

In [ ]:
import importlib
MODULES = [
    ('modules.core.structure',           'Core Models'),
    ('modules.core.handwriting_db',      'HandwritingDB'),
    ('modules.vision.ocr_engine',        'OCR Engine'),
    ('modules.vision.image_preprocessor','Image Preprocessor'),
    ('modules.nlp.arabic_rtl',           'Arabic RTL'),
    ('modules.nlp.arabic_nlp_utils',     'Arabic NLP Utils'),
    ('modules.nlp.reconstruction',       'Reconstruction'),
    ('modules.nlp.feedback',             'Feedback'),
    ('modules.export.exporter',          'Exporter'),
    ('modules.export.layout_preserving', 'Layout Preserving'),
    ('modules.export.markdown_exporter', 'Markdown Exporter'),
    ('modules.evaluation.metrics',       'Metrics'),
    ('modules.security.encryption',      'Encryption'),
]
ok, fail = [], []
print('='*55)
for mod, label in MODULES:
    try:
        importlib.import_module(mod)
        print(f'  OK  {label:<35} {mod}')
        ok.append(mod)
    except Exception as e:
        print(f'  XX  {label:<35} {str(e)[:50]}')
        fail.append(mod)
print('='*55)
pct = len(ok)/len(MODULES)*100
print(f'Health: {len(ok)}/{len(MODULES)} ({pct:.0f}%)')


## 3 — اختبار arabic_nlp_utils

In [ ]:
from modules.nlp.arabic_nlp_utils import normalize_for_comparison, similarity_report

tests = [
    ('arabic-with-diacritics',  'al-tabiibu', 'الطَّبيبُ',  'الطبيب'),
    ('hamza-normalization',     'OCR error',   'الاطباء',   'الأطباء'),
    ('identical',               'exact match', 'مرحبا',     'مرحبا'),
    ('latin-case',              'case insens', 'Hello World','hello world'),
]

print(f'  {"Label":<20} {"Text1":<20} {"Text2":<20} {"Raw":>6} {"Norm":>6} {"OK?"}')
print('-'*85)
for label, desc, t1, t2 in tests:
    r = similarity_report(t1, t2)
    ok_mark = 'PASS' if r['normalized_similarity'] >= 0.9 else 'warn'
    print(f'  {label:<20} {t1:<20} {t2:<20} {r["raw_similarity"]:>6.3f} {r["normalized_similarity"]:>6.3f}  {ok_mark}')


## 4 — اختبار OCR على صور تجريبية

In [ ]:
from PIL import Image, ImageDraw
import numpy as np

img = Image.new('RGB', (500,100), 'white')
draw = ImageDraw.Draw(img)
draw.text((10, 30), 'OmniFile Test - Arabic OCR', fill='black')
img.save('/tmp/test_ocr.png')
display(img)


In [ ]:
# EasyOCR
try:
    import easyocr
    reader = easyocr.Reader(['ar','en'], gpu=False, verbose=False)
    results = reader.readtext('/tmp/test_ocr.png')
    print('EasyOCR results:')
    for bbox, text, conf in results:
        print(f'  [{conf:.1%}] {text}')
    print('EasyOCR: OK')
except Exception as e:
    print(f'EasyOCR: {e}')

# Tesseract
try:
    import pytesseract
    text = pytesseract.image_to_string(img, lang='ara+eng')
    print(f'Tesseract: [{text.strip()[:60]}]')
    print('Tesseract: OK')
except Exception as e:
    print(f'Tesseract: {e}')


## 5 — اختبار التصدير (DOCX + Markdown)

In [ ]:
layout = {
    'blocks': [
        {'type': 'header',    'bbox':[0,0,1,.1],  'text': 'OmniFile v4.2.0 Report'},
        {'type': 'paragraph', 'bbox':[0,.1,1,.5], 'text': 'Test export with table and caption.'},
        {'type': 'table',     'bbox':[0,.5,1,.85],
         'cells': [['Module','Status','Coverage'],
                   ['OCR Engine','OK','95%'],
                   ['NLP Utils','OK','100%'],
                   ['Metrics','OK','98%']]},
        {'type': 'caption',   'bbox':[0,.85,1,.95], 'text': 'Table 1 — Module status'},
    ]
}

# DOCX
try:
    from modules.export.layout_preserving import export_to_docx
    out = export_to_docx(layout, '/tmp/test.docx')
    import os
    print(f'DOCX: {os.path.getsize(out):,} bytes — OK')
except Exception as e:
    print(f'DOCX: {e}')

# Markdown
try:
    from modules.export.markdown_exporter import export_to_markdown
    md_text = export_to_markdown(layout)
    print(f'Markdown ({len(md_text)} chars):')
    print(md_text[:400])
except Exception as e:
    print(f'Markdown: {e}')


## 6 — اختبار مقاييس CER/WER

In [ ]:
try:
    from modules.evaluation.metrics import compute_cer, compute_wer
    pairs = [
        ('hello world',  'hello wrold',  'typo'),
        ('marhaba',      'marhaba',      'identical'),
        ('test Arabic',  'tst Arabik',   'two errors'),
    ]
    print(f'  {"Reference":<20} {"Hypothesis":<20} {"CER":>6} {"WER":>6} {"Grade"}')
    print('-'*65)
    for ref,hyp,note in pairs:
        cer = compute_cer(ref, hyp)
        wer = compute_wer(ref, hyp)
        g = 'A+' if cer<0.02 else 'A' if cer<0.05 else 'B' if cer<0.10 else 'C' if cer<0.20 else 'F'
        print(f'  {ref:<20} {hyp:<20} {cer:>6.3f} {wer:>6.3f}  [{g}] {note}')
    print('Metrics: OK')
except Exception as e:
    print(f'Metrics: {e}')


## 7 — إصلاح أخطاء hf_app.py (3 أخطاء)

**الأخطاء المُصلَحة:**
1. `engine` mismatch: `'Both (Ensemble)'` vs `'Both'` في المعالج
2. تعريف مزدوج لـ `_normalize_text` (يُسبب NameError صامتاً)
3. الإصدار في About: `v1.0` → `v4.2.0`


In [ ]:
hf = open('hf_app.py', encoding='utf-8').read()
orig_len = len(hf)

# Fix 1: engine name mismatch
hf = hf.replace(
    'choices=["EasyOCR", "TrOCR", "Both (Ensemble)"],',
    'choices=["EasyOCR", "TrOCR", "Both"],'
)
print('Fix 1 (engine mismatch): done')

# Fix 2: _normalize_text conflict — rename Tab 6 version
idx1 = hf.find('def _normalize_text')
idx2 = hf.find('def _normalize_text', idx1+1)
if idx2 > -1:
    hf = hf[:idx2] + hf[idx2:].replace('def _normalize_text', 'def _normalize_eval_text', 1)
    hf = hf.replace(
        'ref = _normalize_text(reference)\n    hyp = _normalize_text(hypothesis)',
        'ref = _normalize_eval_text(reference)\n    hyp = _normalize_eval_text(hypothesis)'
    )
    print('Fix 2 (_normalize_text conflict): done')
else:
    print('Fix 2: only one definition found (already fixed or not found)')

# Fix 3: version in About section
hf = hf.replace('OmniFile AI Processor v1.0', 'OmniFile AI Processor v4.2.0')
print('Fix 3 (version): done')

with open('hf_app.py', 'w', encoding='utf-8') as f:
    f.write(hf)
print(f'hf_app.py updated: {orig_len} -> {len(hf)} chars')


## 8 — واجهة Gradio التشخيصية (5 تبويبات)

> `share=True` يعطي رابطاً عاماً يعمل من الجوال لمدة ساعتين.


In [ ]:
import gradio as gr, numpy as np, json, traceback, time, importlib
from PIL import Image

# ── OCR ────────────────────────────────────────────
def run_ocr(image, use_easy, use_tess, langs):
    if image is None: return 'Upload an image', ''
    pil = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    pil.save('/tmp/_in.png')
    out, log = [], []
    t0 = time.time()
    if use_easy:
        try:
            import easyocr
            r = easyocr.Reader(langs, gpu=False, verbose=False)
            res = r.readtext('/tmp/_in.png')
            txt = ' '.join(x[1] for x in res)
            avg = sum(x[2] for x in res)/len(res) if res else 0
            out.append(f'[EasyOCR {avg:.0%}] {txt}')
        except Exception as e: log.append(f'EasyOCR: {e}')
    if use_tess:
        try:
            import pytesseract
            lang_str = '+'.join({'ar':'ara','en':'eng','de':'deu'}.get(l,'eng') for l in langs)
            txt = pytesseract.image_to_string(pil, lang=lang_str)
            out.append(f'[Tesseract] {txt.strip()}')
        except Exception as e: log.append(f'Tess: {e}')
    log.insert(0, f'{time.time()-t0:.1f}s')
    return '\n\n'.join(out) or 'No text', ' | '.join(log)

# ── NLP ─────────────────────────────────────────────
def run_nlp(text, op):
    if not text.strip(): return 'Enter text', ''
    try:
        if op == 'normalize':
            from modules.nlp.arabic_nlp_utils import normalize_for_comparison
            return normalize_for_comparison(text), 'OK'
        elif op == 'similarity':
            lines = text.strip().split('\n')
            if len(lines)<2: return 'Enter 2 lines', ''
            from modules.nlp.arabic_nlp_utils import similarity_report
            r = similarity_report(lines[0], lines[1])
            return json.dumps(r, ensure_ascii=False, indent=2), 'OK'
        elif op == 'detect_lang':
            from langdetect import detect
            return f'Language: {detect(text)}', 'OK'
    except Exception as e:
        return str(e), traceback.format_exc()
    return 'Unknown op', ''

# ── Export ───────────────────────────────────────────
def run_export(text, fmt):
    if not text.strip(): return None, 'Enter text'
    layout = {'blocks': [{'type':'paragraph','bbox':[0,0,1,1],'text':text}]}
    try:
        if fmt == 'DOCX':
            from modules.export.layout_preserving import export_to_docx
            p = '/tmp/out.docx'; export_to_docx(layout, p)
        elif fmt == 'Markdown':
            from modules.export.markdown_exporter import export_to_markdown
            p = '/tmp/out.md'; export_to_markdown(layout, p)
        else:
            p = '/tmp/out.txt'
            open(p,'w',encoding='utf-8-sig').write(text)
        import os
        return p, f'OK — {os.path.getsize(p):,} bytes'
    except Exception as e: return None, str(e)

# ── Metrics ──────────────────────────────────────────
def run_metrics(ref, hyp):
    if not ref.strip() or not hyp.strip(): return 'Enter both texts'
    try:
        from modules.evaluation.metrics import compute_cer, compute_wer
        cer, wer = compute_cer(ref,hyp), compute_wer(ref,hyp)
    except:
        from modules.nlp.arabic_nlp_utils import arabic_normalized_similarity
        return json.dumps({'similarity':round(arabic_normalized_similarity(ref,hyp),4)},indent=2)
    g = 'A+' if cer<0.02 else 'A' if cer<0.05 else 'B' if cer<0.10 else 'C' if cer<0.20 else 'F'
    return json.dumps({'CER':round(cer,4),'WER':round(wer,4),'grade':g},ensure_ascii=False,indent=2)

# ── Health ───────────────────────────────────────────
def run_health():
    mods = ['modules.core.handwriting_db','modules.vision.ocr_engine',
            'modules.nlp.arabic_nlp_utils','modules.nlp.reconstruction',
            'modules.nlp.feedback','modules.export.layout_preserving',
            'modules.export.markdown_exporter','modules.evaluation.metrics',
            'modules.security.encryption']
    lines, ok = ['Module Health Check','='*40], 0
    for m in mods:
        try: importlib.import_module(m); lines.append(f'OK  {m}'); ok+=1
        except Exception as e: lines.append(f'XX  {m}: {str(e)[:50]}')
    pct = ok/len(mods)*100
    lines.append('='*40)
    lines.append(f'Health: {ok}/{len(mods)} ({pct:.0f}%)')
    return '\n'.join(lines)

# ── Build UI ──────────────────────────────────────────
with gr.Blocks(title='OmniFile v4.2.0 Lab',
               theme=gr.themes.Soft(primary_hue='blue',neutral_hue='slate'),
               css='.gradio-container{max-width:900px!important}') as demo:

    gr.HTML('<div style="text-align:center;padding:16px">'
            '<h2 style="margin:0">OmniFile AI Processor v4.2.0</h2>'
            '<p style="color:#888;margin:4px 0">Dr. Abdulmalek | Homs, Syria</p>'
            '</div>')

    with gr.Tab('OCR'):
        with gr.Row():
            with gr.Column(scale=1):
                img_in  = gr.Image(label='Image', type='numpy')
                use_e   = gr.Checkbox(label='EasyOCR', value=True)
                use_t   = gr.Checkbox(label='Tesseract', value=False)
                langs   = gr.CheckboxGroup(['ar','en','de'], value=['ar','en'], label='Languages')
                btn_ocr = gr.Button('Run OCR', variant='primary')
            with gr.Column(scale=2):
                ocr_o = gr.Textbox(label='Result', lines=10, show_copy_button=True)
                ocr_l = gr.Markdown()
        btn_ocr.click(run_ocr, [img_in,use_e,use_t,langs], [ocr_o,ocr_l])

    with gr.Tab('NLP'):
        nlp_i = gr.Textbox(label='Text (2 lines for similarity)', lines=4, rtl=True)
        nlp_op = gr.Radio(['normalize','similarity','detect_lang'], value='normalize', label='Operation')
        gr.Button('Run', variant='primary').click(run_nlp, [nlp_i,nlp_op],
            [gr.Textbox(label='Result',lines=6,rtl=True), gr.Textbox(label='Status',lines=2)])

    with gr.Tab('Export'):
        exp_t = gr.Textbox(label='Text', lines=5, rtl=True)
        exp_f = gr.Radio(['TXT','DOCX','Markdown'], value='TXT', label='Format')
        gr.Button('Export', variant='primary').click(run_export, [exp_t,exp_f],
            [gr.File(label='Download'), gr.Markdown()])

    with gr.Tab('Metrics'):
        with gr.Row():
            ref_t = gr.Textbox(label='Reference', lines=4, rtl=True)
            hyp_t = gr.Textbox(label='OCR Output', lines=4, rtl=True)
        gr.Button('Compute CER/WER', variant='primary').click(run_metrics, [ref_t,hyp_t],
            [gr.Textbox(label='Results',lines=8)])

    with gr.Tab('Health'):
        gr.Button('Check Now', variant='primary', size='lg').click(run_health, [],
            [gr.Textbox(label='Report',lines=20,interactive=False)])

print('Interface ready')


In [ ]:
demo.launch(
    share=True,
    server_name='0.0.0.0',
    server_port=7860,
    show_error=True,
)
# Public URL will appear — open from phone


## 9 — تشغيل الاختبارات

In [ ]:
!cd /content/OmniFile_Processor && \
  pip install -q pytest pytest-cov 2>/dev/null && \
  python -m pytest tests/test_arabic_nlp_utils.py \
                   tests/test_layout_preserving.py \
                   tests/test_metrics.py \
  -v --tb=short 2>&1 | head -60


## 10 — تقييم KITAB-Bench (وضع تجريبي)

In [ ]:
!python scripts/benchmark_kitab.py --demo


## 11 — رفع إصلاح hf_app.py إلى GitHub

In [ ]:
import subprocess

TOKEN = ''  # <-- ضع GitHub token هنا

if TOKEN:
    subprocess.run(['git','add','hf_app.py'], cwd='/content/OmniFile_Processor')
    subprocess.run(
        ['git','commit','-m','fix: hf_app.py bugs — engine mismatch, normalize conflict, version [v4.2.1]'],
        cwd='/content/OmniFile_Processor'
    )
    result = subprocess.run(
        ['git','push', f'https://{TOKEN}@github.com/DrAbdulmalek/OmniFile_Processor.git', 'main'],
        cwd='/content/OmniFile_Processor', capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print('Set TOKEN above to push to GitHub')
    print('Or run locally: git add hf_app.py && git commit -m fix && git push')
